# 2022 SE QLD Floods — Entity Liability

Same corrected pipeline as Black Summer: **global-share** apportionment, PR/FAR from the
multiplicative GEV shift-fit (notebook 07), uncertainty from the PR bootstrap.

- **Primary PR ≈ 1.11, FAR ≈ 10%** (CC 7%/°C × α_QLD=0.289 — conservative lower bound).
- **Damage central AUD 10B remains a placeholder** pending an official QLD Treasury / Deloitte
  / NEMA figure; EM-DAT records a redacted value for REDACTED-DISNO, consistent with that order.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
from src.attribution import (
    area_weighted_series, season_block_max, wet_season_max_ndays,
    load_gmst, extrapolate_to, smoothed_covariate, event_gmst_sigma,
    shift_fit_gev, fit_gev, build_liability_table, far,
    AUD_TO_USD, CC_RATE_STANDARD, CC_RATE_HIGH, CLIM_START, CLIM_END,
)
from scipy.stats import genextreme

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
ew = pd.read_parquet(PROC / 'entity_warming_contribution.parquet')
print(f'Carbon Majors coverage: {ew["global_share"].sum()*100:.1f}%')

pr_df = pd.read_csv(PROC / 'qld_floods_pr_era5.csv')
primary_row = pr_df[pr_df['method'].str.startswith('primary')].iloc[0]
PR_PRIMARY = float(primary_row['pr'])
boot = pd.read_parquet(PROC / 'qld_floods_pr_shiftfit_bootstrap.parquet')['pr_boot'].values
print(f'Primary PR = {PR_PRIMARY:.3f}, FAR = {far(PR_PRIMARY):.3f}  (bootstrap n={len(boot)})')

FX = AUD_TO_USD[2022]
scenarios = {
    'conservative':  dict(damages_usd_b=5.56 * FX, pr=PR_PRIMARY, pr_samples=boot,
                          label='ICA insured, AUD 5.56B'),
    'central':       dict(damages_usd_b=10.0 * FX, pr=PR_PRIMARY, pr_samples=boot,
                          label='Direct economic (placeholder), AUD 10B'),
    'comprehensive': dict(damages_usd_b=20.0 * FX, pr=PR_PRIMARY, pr_samples=boot,
                          label='Social cost (placeholder), AUD 20B'),
}


In [ ]:
liability, totals = build_liability_table(ew, scenarios)
liability.to_parquet(PROC / 'qld_floods_liability.parquet', index=False)
totals.to_csv(PROC / 'qld_floods_scenario_totals.csv', index=False)

print('Scenario totals (Carbon Majors, ~54% of global):')
print(totals[['scenario', 'damages_usd_b', 'far', 'far_p05', 'far_p95',
              'total_attributed_usd_b']].to_string(index=False))
print('\nTop 10 entities — central scenario (USD M):')
cols = ['rank', 'parent_entity', 'parent_type', 'liability_central_USD_M',
        'liability_central_p05_USD_M', 'liability_central_p95_USD_M']
top10 = liability.head(10)[cols].copy()
for c in cols[3:]:
    top10[c] = top10[c].map('{:,.2f}'.format)
print(top10.to_string(index=False))
ar = liability.loc[liability.parent_entity == 'Saudi Aramco']
print(f'\nSaudi Aramco central: USD {ar["liability_central_USD_M"].iloc[0]:.1f}M')


In [ ]:
# ── Figure: PR × damages sensitivity grid (central uncertainty is small here) ──
aramco = float(liability.loc[liability['parent_entity'] == 'Saudi Aramco', 'global_share'].iloc[0])
pr_range = [1.1, 1.2, 1.4, 1.8, 2.5, 4.0]
dmg_aud  = [5.56, 10, 15, 20, 30, 50]
grid = pd.DataFrame(
    index=[f'PR={p}' for p in pr_range],
    columns=[f'AUD {d}B' for d in dmg_aud],
    data=[[aramco * far(p) * d * FX * 1000 for d in dmg_aud] for p in pr_range])
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(grid.astype(float), annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label': 'USD millions'}, linewidths=0.5, ax=ax)
ax.set_title('Saudi Aramco — 2022 QLD Floods liability sensitivity (USD M)', fontsize=11)
ax.set_xlabel('Total damages'); ax.set_ylabel('Probability Ratio')
plt.tight_layout(); plt.savefig(FIGS / 'qld_floods_sensitivity_aramco.png', bbox_inches='tight')
plt.show()
print(f'Aramco global warming share: {aramco*100:.3f}%')
